In [0]:
alphacollector_response=dbutils.widgets.get("alphacollector_response")
volume_path=dbutils.widgets.get("volume_path")
control_table = dbutils.widgets.get("control_table")

In [0]:
import re
import os
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp

def to_snake_case(name):
    """Convert column names to snake_case format"""
    name = name.strip().replace(' ', '_')
    name = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub('([a-z0-9])([A-Z])', r'\1_\2', name)
    name = name.lower()
    name = re.sub('_+', '_', name)
    return name

def extract_file_metadata(file_path):
    """Extract metadata from file path"""
    file_name = os.path.basename(file_path)
    
    try:
        file_info = dbutils.fs.ls(file_path)[0]
        file_size = file_info.size
    except:
        file_size = 0
    
    return {
        'file_name': file_name,
        'file_path': file_path,
        'file_size': file_size
    }

def is_file_already_processed(file_name, control_table):
    """Check if file has already been processed"""
    try:
        result = spark.sql(f"""
            SELECT COUNT(*) as count 
            FROM {control_table}
            WHERE file_name = '{file_name}' 
            AND status = 'SUCCESS'
        """).collect()[0]['count']
        
        return result > 0
    except Exception as e:
        print(f"Control table check failed: {e}")
        return False

def log_file_processing(file_metadata, record_count, control_table, status='SUCCESS'):
    """Log file processing to control table"""
    try:
        job_run_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().currentRunId().toString()
    except:
        job_run_id = "unknown"
    
    file_name_escaped = file_metadata['file_name'].replace("'", "''")
    file_path_escaped = file_metadata['file_path'].replace("'", "''")
    
    insert_sql = f"""
    INSERT INTO {control_table} 
    VALUES (
        '{file_name_escaped}',
        '{file_path_escaped}',
        {file_metadata['file_size']},
        {record_count},
        current_timestamp(),
        '{job_run_id}',
        '{status}'
    )
    """
    
    spark.sql(insert_sql)

def get_latest_file_in_volume(volume_path):
    """Get the most recently modified file in the volume"""
    try:
        files = dbutils.fs.ls(volume_path)
        data_files = [f for f in files if f.name.endswith(('.txt', '.csv')) or 'Bayada_Notes_' in f.name]
        
        if not data_files:
            raise FileNotFoundError(f"No data files found in {volume_path}")
        latest_file = sorted(data_files, key=lambda x: x.modificationTime, reverse=True)[0]
        return latest_file.path
    except Exception as e:
        print(f"Error getting latest file: {e}")
        raise
print("ALPHACOLLECTOR RESPONSE - FILE PROCESSING")


In [0]:
triggered_file = get_latest_file_in_volume(volume_path)
file_metadata = extract_file_metadata(triggered_file)

print(f"\nTriggered File: {file_metadata['file_name']}")
print(f"File Path: {file_metadata['file_path']}")
print(f"File Size: {file_metadata['file_size']:,} bytes")

if is_file_already_processed(file_metadata['file_name'], control_table):
    print(f"\n⚠️  FILE ALREADY PROCESSED")
    print(f"File '{file_metadata['file_name']}' has already been processed.")
    print("Skipping to prevent duplicates.")
    print("=" * 80)
    
    last_processing = spark.sql(f"""
        SELECT processing_timestamp, records_processed, job_run_id
        FROM {control_table}
        WHERE file_name = '{file_metadata['file_name']}'
        ORDER BY processing_timestamp DESC
        LIMIT 1
    """).collect()[0]
    
    print(f"\nLast Processed: {last_processing['processing_timestamp']}")
    print(f"Records Processed: {last_processing['records_processed']}")
    print(f"Job Run ID: {last_processing['job_run_id']}")
 
    dbutils.notebook.exit("File already processed - skipped")
else:
    print(f"\n✓ NEW FILE DETECTED")
    print("Proceeding with processing...")
  

In [0]:
try:
    # Read the specific file with pipe delimiter
    df_alphacollector_repsone = spark.read.option(
        "header", "true"
    ).option(
        "delimiter", "|"  
    ).csv(triggered_file)
    
    # Convert column names to snake_case
    for old_col in df_alphacollector_repsone.columns:
        new_col = to_snake_case(old_col)
        df_alphacollector_repsone = df_alphacollector_repsone.withColumnRenamed(old_col, new_col)
    
    # Get record count before processing
    record_count = df_alphacollector_repsone.count()
    
    df_alphacollector_repsone.createOrReplaceTempView("assignment_domain_view")
    
    print("\n" + "=" * 80)
    print("DATA PREVIEW")
    print("=" * 80)
    print(f"Column names after conversion: {df_alphacollector_repsone.columns}")
    print(f"Total records read: {record_count:,}")
    print("=" * 80)
    
    display(df_alphacollector_repsone)

except Exception as e:
    print(f"\n❌ ERROR READING FILE: {e}")
    log_file_processing(file_metadata, 0, control_table, status='FAILED')
    raise


In [0]:
try:
    df_alphacollector_repsone.write.mode("append").insertInto(alphacollector_response)
    print("✓ DATA SUCCESSFULLY INSERTED")
    print(f"Table: {alphacollector_response}")
    print(f"Records Inserted: {record_count:,}")
    
    log_file_processing(file_metadata, record_count, control_table, status='SUCCESS')
    
    print("\n✓ Processing logged to control table")
    print("=" * 80)
    
except Exception as e:
    print(f"\n❌ ERROR INSERTING DATA: {e}")
    log_file_processing(file_metadata, record_count, control_table, status='FAILED')
    raise